<a href="https://colab.research.google.com/github/riyap05/yolov8-gpu-inference-optimization/blob/main/yolov8_gpu_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi
!pip install -q ultralytics
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

Sat Nov  7 12:25:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import cv2, urllib.request, time, torch
from ultralytics import YOLO

urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg", "bus.jpg")
img = cv2.imread("bus.jpg")
images = [img] * 64          # 64 copies of the same image

model = YOLO("yolov8n.pt")   # small pretrained YOLOv8 model
print("Loaded", len(images), "images of shape", img.shape)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Loaded 64 images of shape (1080, 810, 3)


In [3]:
def benchmark(model, images, batch=1, half=False, runs=3):
    for _ in range(2):   # warm-up runs, not timed
        model.predict(images[:batch], half=half, verbose=False)
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(runs):
        for i in range(0, len(images), batch):
            model.predict(images[i:i+batch], half=half, verbose=False)
    torch.cuda.synchronize()
    total = time.perf_counter() - start
    n = runs * len(images)
    return 1000 * total / n, n / total   # ms per image, images per second

In [4]:
ms, fps = benchmark(model, images, batch=1, half=False)
print(f"Baseline FP32, batch 1: {ms:.2f} ms/image, {fps:.1f} images/sec")

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.

In [5]:
import logging
logging.getLogger("ultralytics").setLevel(logging.ERROR)   # hide warnings

def benchmark(model, images, batch=1, half=False, runs=3):
    kw = {"half": True} if half else {}   # only pass half when needed
    for _ in range(2):
        model.predict(images[:batch], verbose=False, **kw)
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(runs):
        for i in range(0, len(images), batch):
            model.predict(images[i:i+batch], verbose=False, **kw)
    torch.cuda.synchronize()
    total = time.perf_counter() - start
    n = runs * len(images)
    return 1000 * total / n, n / total

In [6]:
ms16, fps16 = benchmark(model, images, batch=1, half=True)
print(f"FP16, batch 1: {ms16:.2f} ms/image, {fps16:.1f} images/sec")
print(f"Speedup vs baseline: {11.04 / ms16:.2f}x")

FP16, batch 1: 9.56 ms/image, 104.6 images/sec
Speedup vs baseline: 1.15x


In [7]:
r32 = model.predict(img, verbose=False)[0]
r16 = model.predict(img, half=True, verbose=False)[0]
print("FP32:", len(r32.boxes), "objects, confidences:", r32.boxes.conf.cpu().numpy().round(3))
print("FP16:", len(r16.boxes), "objects, confidences:", r16.boxes.conf.cpu().numpy().round(3))
print("FP32 time breakdown (ms):", r32.speed)
print("FP16 time breakdown (ms):", r16.speed)

FP32: 6 objects, confidences: [      0.873       0.866       0.853       0.825       0.261       0.255]
FP16: 6 objects, confidences: [      0.873       0.866       0.853       0.825       0.261       0.255]
FP32 time breakdown (ms): {'preprocess': 2.964095000038469, 'inference': 10.520843999984208, 'postprocess': 3.7013229999729447}
FP16 time breakdown (ms): {'preprocess': 1.836105999927895, 'inference': 6.625406000011935, 'postprocess': 1.1839310000141268}


In [8]:
for half in [False, True]:
    for b in [1, 8, 16, 32, 64]:
        torch.cuda.reset_peak_memory_stats()
        ms, fps = benchmark(model, images, batch=b, half=half)
        mem = torch.cuda.max_memory_allocated() / 1024**2
        p = "FP16" if half else "FP32"
        print(f"{p} batch {b:>2}: {ms:6.2f} ms/img, {fps:7.1f} img/s, "
              f"{ms*b:7.1f} ms/batch, {mem:6.0f} MB GPU memory")

FP32 batch  1:  10.55 ms/img,    94.7 img/s,    10.6 ms/batch,     42 MB GPU memory
FP32 batch  8:   5.19 ms/img,   192.8 img/s,    41.5 ms/batch,    232 MB GPU memory
FP32 batch 16:   5.09 ms/img,   196.5 img/s,    81.4 ms/batch,    333 MB GPU memory
FP32 batch 32:   6.55 ms/img,   152.6 img/s,   209.7 ms/batch,    643 MB GPU memory
FP32 batch 64:   6.64 ms/img,   150.5 img/s,   425.1 ms/batch,   1260 MB GPU memory
FP16 batch  1:   9.84 ms/img,   101.7 img/s,     9.8 ms/batch,     34 MB GPU memory
FP16 batch  8:   4.20 ms/img,   238.2 img/s,    33.6 ms/batch,     95 MB GPU memory
FP16 batch 16:   3.98 ms/img,   251.4 img/s,    63.7 ms/batch,    166 MB GPU memory
FP16 batch 32:   4.38 ms/img,   228.5 img/s,   140.0 ms/batch,    307 MB GPU memory
FP16 batch 64:   4.36 ms/img,   229.1 img/s,   279.3 ms/batch,    590 MB GPU memory


In [9]:
trt_path = model.export(format="engine", half=True, dynamic=True, batch=16)
print("Saved TensorRT engine to:", trt_path)

2026-11-07 12:43:35,425 - [modelopt][onnx] - WARNING - Shared constants were detected and duplicated accordingly.


2026-11-07 12:43:35,618 - [modelopt][onnx] - INFO - Successfully enabled 1 EPs for ORT: ['CPUExecutionProvider']


INFO:modelopt.onnx:Successfully enabled 1 EPs for ORT: ['CPUExecutionProvider']


2026-11-07 12:43:36,244 - [modelopt][onnx] - INFO - Running ONNX Runtime to obtain reference outputs (this may take a while)...


INFO:modelopt.onnx.autocast:Running ONNX Runtime to obtain reference outputs (this may take a while)...


2026-11-07 12:43:48,219 - [modelopt][onnx] - INFO - Skipping node /model.22/Mul: reference IO out of range: min=6400, max=6400, absmax=6400, range=[-512, 512]


... (35 outputs left out)

2026-11-07 12:43:49,276 - [modelopt][onnx] - WARNING - Did not find  in value info map! Assuming not castable


2026-11-07 12:43:51,358 - [modelopt][onnx] - INFO - Converted 297/321 nodes (92.52%) to fp16


INFO:modelopt.onnx.autocast:Converted 297/321 nodes (92.52%) to fp16


Saved TensorRT engine to: yolov8n.engine


In [10]:
trt_model = YOLO(trt_path, task="detect")
for b in [1, 8, 16]:
    ms, fps = benchmark(trt_model, images, batch=b)
    print(f"TensorRT FP16 batch {b:>2}: {ms:6.2f} ms/img, {fps:7.1f} img/s, "
          f"speedup vs baseline: {11.04/ms:.2f}x")

TensorRT FP16 batch  1:   7.83 ms/img,   127.7 img/s, speedup vs baseline: 1.41x
TensorRT FP16 batch  8:   4.35 ms/img,   229.6 img/s, speedup vs baseline: 2.54x
TensorRT FP16 batch 16:   4.50 ms/img,   222.1 img/s, speedup vs baseline: 2.45x


In [11]:
import numpy as np
for name, m, kw in [("PyTorch FP16", model, {"half": True}),
                    ("TensorRT FP16", trt_model, {})]:
    m.predict(images[:16], verbose=False, **kw)   # warm-up
    runs = [m.predict(images[:16], verbose=False, **kw)[0].speed for _ in range(10)]
    avg = {k: round(np.mean([r[k] for r in runs]), 2) for k in runs[0]}
    print(f"{name}, batch 16 (ms per image): {avg}")

PyTorch FP16, batch 16 (ms per image): {'preprocess': np.float64(3.09), 'inference': np.float64(2.22), 'postprocess': np.float64(2.01)}
TensorRT FP16, batch 16 (ms per image): {'preprocess': np.float64(1.63), 'inference': np.float64(0.95), 'postprocess': np.float64(1.01)}
